# Endless Fiction on Colab

Runs this repository's lncrawl server in a free Colab runtime.

* The library, glossary, keys and config live in **`MyDrive/data`**, so they survive a
disconnect, a restart and a new runtime.
* Cloudflare hands the server a public HTTPS address — no account, token or tunnel setup.
* The Admin Dashboard opens at `<url>/admin`; the reader and library are at `<url>/`.

Run the first cell and click the link it prints. Keep it running: when the notebook stops,
the server stops with it. If the repository is private, save a GitHub token with read
access to it as the Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) before the
first cell; the clone uses it and never writes it to disk.


In [ ]:
#@title 1 · Install, start the server, get a Cloudflare link (first run ≈4 min) { display-mode: "form" }
# The data folder in your Drive. Everything the server writes ends up here.
DATA_DIR = "/content/drive/MyDrive/data"

REPO = "https://github.com/AzureDragon84/EndlessFiction"
BRANCH = "project-alpha"  # the branch this fork's work lives on
APP_DIR = "/content/EndlessFiction"
PORT = "3001"

import base64, getpass, os, re, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)


def github_token():
    """A read token for a private repo: Colab secret first, then a hidden prompt."""
    try:
        from google.colab import userdata
        value = userdata.get("GITHUB_TOKEN")
        if value:
            return value.strip()
    except Exception:
        pass
    try:
        return getpass.getpass("GitHub token with read access to this repo (blank if public): ").strip()
    except (EOFError, OSError):
        return ""


def clone(token=""):
    """Clone the branch; a token travels in a one-shot header, never in .git/config."""
    shutil.rmtree(APP_DIR, ignore_errors=True)
    command = ["git"]
    if token:
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        command += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]
    print(f"Cloning {BRANCH} from {REPO}...")
    return subprocess.run(
        [*command, "clone", "--depth", "1", "--branch", BRANCH, REPO, APP_DIR],
        capture_output=True,
        text=True,
    )


app = Path(APP_DIR)
if (app / ".git").is_dir():
    print("Repository already cloned; reusing", app)
else:
    done = clone()
    if done.returncode and (token := github_token()):
        print("Retrying with the token from GITHUB_TOKEN...")
        done = clone(token)
    if done.returncode:
        shutil.rmtree(app, ignore_errors=True)
        raise SystemExit(
            "git clone failed:\n\n"
            + (done.stderr or done.stdout or "").strip()
            + "\n\nThis repository is private. Save a GitHub token with read access to it as "
            "the Colab secret GITHUB_TOKEN (key icon in the left sidebar), or make the "
            "repository public, then run this cell again."
        )

print("Installing the package and its dependencies…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(app)], check=True)

binary = Path("/usr/local/bin/cloudflared")
if not binary.exists():
    subprocess.run(
        f"wget -q -O {binary} "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        f" && chmod +x {binary}",
        shell=True,
        check=True,
    )

# A previous run in this runtime may have left the old processes behind.
for name in ("server", "tunnel"):
    pid_file = Path(f"/content/{name}.pid")
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text()), 15)
        except (ProcessLookupError, ValueError):
            pass
        pid_file.unlink()

env = {**os.environ, "LNCRAWL_DATA_PATH": DATA_DIR}
server = subprocess.Popen(
    [sys.executable, "-m", "lncrawl", "server", "--port", PORT, "--host", "0.0.0.0"],
    cwd=str(app),
    env=env,
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
Path("/content/server.pid").write_text(str(server.pid))

print("Waiting for the server to answer /health…")
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError("The server exited during startup; see /content/server.log")
    probe = subprocess.run(
        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", f"http://127.0.0.1:{PORT}/health"],
        capture_output=True,
        text=True,
    )
    if probe.stdout.strip() == "200":
        break
    time.sleep(2)
else:
    raise RuntimeError("The server never answered /health; see /content/server.log")

print("Opening the Cloudflare tunnel…")
tunnel = subprocess.Popen(
    [str(binary), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open("/content/tunnel.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
Path("/content/tunnel.pid").write_text(str(tunnel.pid))

url = None
for _ in range(60):
    found = re.findall(
        r"https://[a-z0-9-]+\.trycloudflare\.com", Path("/content/tunnel.log").read_text()
    )
    if found:
        url = found[0]
        break
    if tunnel.poll() is not None:
        print(Path("/content/tunnel.log").read_text()[-2000:])
        raise RuntimeError("cloudflared exited before it printed a URL")
    time.sleep(2)

from IPython.display import HTML, display

if url:
    display(
        HTML(
            f'<h3><a href="{url}/admin" target="_blank">Open the Admin Dashboard</a> '
            f'&nbsp;·&nbsp; <a href="{url}/" target="_blank">Library &amp; reader</a></h3>'
            f'<p><code>{url}</code></p>'
            f'<p style="color:#666">Sign in as <b>admin_alpha</b> / <b>admin_alpha</b> '
            f'(the login field is labelled Email), then change the password.</p>'
        )
    )
else:
    print("cloudflared did not print a URL yet; read /content/tunnel.log")

print(f"Server data: {DATA_DIR}")


## Using it

The **Admin Dashboard** (`/admin`) owns the glossary, the key pool and chapter translation;
the library at `/` browses, downloads and reads.

* **Glossary extraction → Decide batch** spends one paid call per 50 terms, and the free
  local scan (jieba for Chinese) runs alongside each paid extract batch, so names the model
  skipped still reach the review sheet.
* The **Review** table shows each term's category and the chapter it first appears in, and
  is filterable by category, so `special` can be drained into `character`, `rank_realm` and
  the rest.
* Everything is stored in `MyDrive/data/sqlite.db`; translation output lives beside it.

The link is a `trycloudflare.com` hostname that only exists while `cloudflared` runs. Rerun
the first cell after a disconnect and it prints a new one.


In [ ]:
#@title 2 · Stop the server and the tunnel
import os
from pathlib import Path

for name in ("tunnel", "server"):
    pid_file = Path(f"/content/{name}.pid")
    if not pid_file.exists():
        continue
    try:
        os.kill(int(pid_file.read_text()), 15)
        print(f"Stopped {name}.")
    except (ProcessLookupError, ValueError):
        print(f"{name} was not running.")
    pid_file.unlink()

print("Your data is still in /content/drive/MyDrive/data.")
